In [2]:
# 1.2
# 1. import the necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp

# 2. import csv
data = pd.read_csv('./data_and_materials/gamma-ray.csv')
# check the data
data.head()


,seconds,count
0,116.0,0.0
1,112.0,0.0
2,160.0,0.0
3,51.5,0.0
4,102.0,1.0


## Problem 1.2, part (d)

Under the null model, the emission rate is constant:

$$G_i \sim \mathrm{Poisson}(\lambda t_i).$$

The likelihood is

$$L(\lambda)=\prod_i \frac{e^{-\lambda t_i}(\lambda t_i)^{G_i}}{G_i!}.$$

The log-likelihood, up to constants that do not depend on $\lambda$, is

$$\ell(\lambda)=-\lambda\sum_i t_i + \left(\sum_i G_i\right)\log \lambda + C.$$

Differentiate and set the derivative equal to zero:

$$\frac{d\ell}{d\lambda}=-\sum_i t_i + \frac{\sum_i G_i}{\lambda}=0.$$

Therefore the MLE is

$$\hat\lambda=\frac{\sum_i G_i}{\sum_i t_i}.$$


In [3]:
total_count = data['count'].sum()
total_time = data['seconds'].sum()
lambda_hat_null = total_count / total_time

print(f"sum G_i = {total_count:g}")
print(f"sum t_i = {total_time:g}")
print(f"lambda_hat = {lambda_hat_null}")
print(f"3 significant figures: {lambda_hat_null:.3g}")

sum G_i = 61
sum t_i = 15718.2
lambda_hat = 0.0038808514969907496
3 significant figures: 0.00388


For the answer box in part (d), enter

$$\boxed{0.00388}$$

This is a rate per second. It is not the total number of observed gamma rays.

## Problem 1.2, part (e)

Under the alternative model, each interval has its own rate:

$$G_i \sim \mathrm{Poisson}(\lambda_i t_i).$$

For one interval, the log-likelihood is

$$\ell_i(\lambda_i)=-\lambda_i t_i + G_i\log(\lambda_i t_i)-\log(G_i!).$$

Keeping only terms involving $\lambda_i$:

$$\ell_i(\lambda_i)=-\lambda_i t_i + G_i\log\lambda_i + C.$$

Differentiate and set equal to zero:

$$\frac{d\ell_i}{d\lambda_i}=-t_i+\frac{G_i}{\lambda_i}=0.$$

So the MLE for each interval is

$$\hat\lambda_i=\frac{G_i}{t_i}.$$

For the answer box in part (e), enter `G_i/t_i`.

In [ ]:
data['lambda_hat_i'] = data['count'] / data['seconds']
data.head()

## Problem 1.4

这一题的目标是：对 3051 个基因分别做一次 ALL vs AML 的两样本 Welch t-test，然后统计在显著性水平 $\alpha=0.05$ 下有多少个基因显著。

对第 $i$ 个基因，原假设和备择假设是

$$H_{0,i}: \mu_{\mathrm{ALL},i}=\mu_{\mathrm{AML},i}, \qquad H_{1,i}: \mu_{\mathrm{ALL},i}\ne\mu_{\mathrm{AML},i}.$$

因为题目给出的统计量是 Welch unequal variances t-test，所以不假设 ALL 和 AML 两组方差相同。每个基因都会得到一个双侧 p-value。Part (a) 直接数未校正 p-value 不超过 0.05 的基因数；Part (b) 先对 3051 个 p-value 做多重检验校正，再数校正后显著的基因数。

In [ ]:
# Problem 1.4: load Golub gene expression data
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

golub_dir = Path('data_and_materials/golub_data')
golub = pd.read_csv(golub_dir / 'golub.csv', index_col=0)
golub_class = pd.read_csv(golub_dir / 'golub_cl.csv', index_col=0)['x']

all_samples = golub.loc[:, golub_class.to_numpy() == 0]
aml_samples = golub.loc[:, golub_class.to_numpy() == 1]

print('golub shape:', golub.shape)
print('ALL samples:', all_samples.shape[1])
print('AML samples:', aml_samples.shape[1])

### Welch t-test 的计算

对每个基因，先计算两组样本均值、样本方差和标准误：

$$SE_i=\sqrt{\frac{s^2_{\mathrm{ALL},i}}{N_{\mathrm{ALL}}}+\frac{s^2_{\mathrm{AML},i}}{N_{\mathrm{AML}}}}.$$

Welch 统计量是

$$t_i=\frac{\bar X_{\mathrm{ALL},i}-\bar X_{\mathrm{AML},i}}{SE_i}.$$

自由度使用题目给出的 Welch-Satterthwaite 近似。因为题目问表达水平是否不同，所以使用双侧 p-value：

$$p_i=2P(T_{\nu_i}\ge |t_i|).$$

In [ ]:
# Manual Welch t-test, following the formulas in the problem statement
n_all = all_samples.shape[1]
n_aml = aml_samples.shape[1]

mean_all = all_samples.mean(axis=1)
mean_aml = aml_samples.mean(axis=1)
var_all = all_samples.var(axis=1, ddof=1)
var_aml = aml_samples.var(axis=1, ddof=1)

se = np.sqrt(var_all / n_all + var_aml / n_aml)
t_welch = (mean_all - mean_aml) / se

df_welch = (var_all / n_all + var_aml / n_aml) ** 2 / (
    (var_all / n_all) ** 2 / (n_all - 1)
    + (var_aml / n_aml) ** 2 / (n_aml - 1)
)

p_values = 2 * stats.t.sf(np.abs(t_welch), df_welch)

# This should match scipy.stats.ttest_ind(..., equal_var=False).
scipy_p_values = stats.ttest_ind(
    all_samples,
    aml_samples,
    axis=1,
    equal_var=False,
    nan_policy='omit'
).pvalue

print('max absolute difference from scipy:', np.max(np.abs(p_values - scipy_p_values)))

### Parts (a) and (b)

未校正时，直接计算 $p_i\le 0.05$ 的个数。

Holm-Bonferroni 是控制 FWER 的 step-down 方法，通常比普通 Bonferroni 略不保守。Benjamini-Hochberg 是控制 FDR 的方法，通常会比 Holm-Bonferroni 给出更多拒绝。这里用 `statsmodels.stats.multitest.multipletests` 对同一批 p-value 做校正。

In [ ]:
alpha = 0.05

num_uncorrected = int((p_values <= alpha).sum())

reject_holm, p_holm, _, _ = multipletests(p_values, alpha=alpha, method='holm')
reject_bh, p_bh, _, _ = multipletests(p_values, alpha=alpha, method='fdr_bh')

num_holm = int(reject_holm.sum())
num_bh = int(reject_bh.sum())

print('Part (a), uncorrected p-values:', num_uncorrected)
print('Part (b), Holm-Bonferroni:', num_holm)
print('Part (b), Benjamini-Hochberg:', num_bh)

所以答题框中应填写：

- Part (a), uncorrected p-values: $\boxed{1078}$
- Part (b), Holm-Bonferroni: $\boxed{103}$
- Part (b), Benjamini-Hochberg: $\boxed{695}$

### Part (c)

答案是 **Yes**。

把 p-value 从小到大排序为 $p_{(1)}\le\cdots\le p_{(m)}$。Holm-Bonferroni 在第 $k$ 个位置使用的临界值是

$$\frac{\alpha}{m-k+1},$$

而 Benjamini-Hochberg 在第 $k$ 个位置使用的临界值是

$$\frac{k\alpha}{m}.$$

对所有 $k=1,\ldots,m$，都有

$$\frac{\alpha}{m-k+1}\le \frac{k\alpha}{m},$$

因为这等价于 $m\le k(m-k+1)$，也就是 $(k-1)(m-k)\ge0$。因此只要某个排序位置能通过 Holm-Bonferroni 的阈值，它也不会超过 Benjamini-Hochberg 对应位置的阈值。所以在同一批数据、同一个 $\alpha$ 下，BH 的拒绝数总是至少和 Holm-Bonferroni 一样多。